In [1]:
import sys
from pathlib import Path

from chromadb.api.client import SharedSystemClient

# Reset process-local clients after the persistent index was removed.
SharedSystemClient.clear_system_cache()

project_root = Path.cwd()
if not (project_root / "src").exists():
    for parent in project_root.resolve().parents:
        if (parent / "src").exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.core.config import Config
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.embedder import Embedder
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import Reranker
from src.retrieval.vector_store import VectorStore

config = Config()

In [2]:
# Load chunks already produced by src/document (parsing + chunking)
import json

chunks_path = project_root / "output/v2/chunks/all_chunks.json"
with chunks_path.open("r", encoding="utf-8") as chunks_file:
    chunks = json.load(chunks_file)
print(f"Loaded {len(chunks)} chunk(s) from {chunks_path}")

# Encode embeddings
embedder = Embedder(config)
chunk_embeddings = embedder.embed_chunks(chunks)
for chunk, embedding in zip(chunks, chunk_embeddings):
    chunk["embedding"] = embedding

# Store chunks in ChromaDB
vector_store = VectorStore(config, embedder=embedder)
vector_store.add_chunks(chunks)

# Build BM25 index
bm25 = BM25Retriever(chunks, config)

# Run retrieval
query = "What is the classification accuracy of ResNet26-V2?"
vector_results = vector_store.search(query, top_k=20)
bm25_results = bm25.search(query, top_k=20)

print("Vector results:")
for result in vector_results:
    print(result)

print("BM25 results:")
for result in bm25_results:
    print(result)


Loaded 306 chunk(s) from /Users/humengqing/Documents/Code/VSCode/doc-qa-agent/output/v2/chunks/all_chunks.json


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Vector results:
{'chunk_id': 'HuMengqing_chunk_126', 'text': 'By comparing the results, it is found that ResNet26-V2 performs better in each metric relative to EfficientNet-B0 and VGG16. In addition, the ResNet26-V2 model designed in this study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary classification tasks, has about 5.3 million parameters and VGG16 has 138 million parameters. It is worth noting that ResNet26-V2 achieves better performance with far fewer parameters than the other models, indicating that it significantly improves the computational efficiency and memory footprint while maintaining the accuracy of the model. This makes ResNet26-V2 a better choice for binary classification tasks in practical applications, especially in resource-limited environments.', 'metadata': {'section_title': '4.4 Comparison with Other Convolutional Neural Networks', 'chunk_index': 126, 'page': 63, 'chunk_type': 'text', 'source': 'HuMengqing.pdf'}, 'distance':

In [3]:
hybrid_retriever = HybridRetriever(vector_store, bm25, config)
hybrid_results = hybrid_retriever.search(query)

print(f"Hybrid results: {len(hybrid_results)}")
for rank, result in enumerate(hybrid_results, start=1):
    print(f"\n{rank}. {result['chunk_id']} | RRF: {result['rrf_score']:.6f}")
    print(
        f"Dense rank: {result['dense_rank']} | "
        f"BM25 rank: {result['bm25_rank']}"
    )
    print(result['metadata'])
    print(result['text'][:300])


Hybrid results: 10

1. HuMengqing_chunk_126 | RRF: 0.032787
Dense rank: 1 | BM25 rank: 1
{'source': 'HuMengqing.pdf', 'chunk_index': 126, 'chunk_type': 'text', 'page': 63, 'section_title': '4.4 Comparison with Other Convolutional Neural Networks'}
By comparing the results, it is found that ResNet26-V2 performs better in each metric relative to EfficientNet-B0 and VGG16. In addition, the ResNet26-V2 model designed in this study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary classification tasks, has about 5

2. HuMengqing_chunk_132 | RRF: 0.031514
Dense rank: 5 | BM25 rank: 2
{'page': 64, 'chunk_index': 132, 'section_title': '5 Summary', 'chunk_type': 'text', 'source': 'HuMengqing.pdf'}
To verify the effectiveness of the ResNet-V2 model, comparative experiments are conducted with other classical convolutional neural network models, including EfficientNet-B0 and VGG16. The experimental results show that the ResNet26-V2 model not only outperforms the o

In [4]:
reranker = Reranker(config)
reranked_results = reranker.rerank(query, hybrid_results)

print(f"Reranked results: {len(reranked_results)}")
for rank, result in enumerate(reranked_results, start=1):
    print(
        f"\n{rank}. {result['chunk_id']} | "
        f"Rerank: {result['rerank_score']:.6f}"
    )
    print(f"RRF: {result['rrf_score']:.6f}")
    print(result['metadata'])
    print(result['text'][:300])

Reranked results: 5

1. HuMengqing_chunk_132 | Rerank: 0.983888
RRF: 0.031514
{'page': 64, 'chunk_index': 132, 'section_title': '5 Summary', 'chunk_type': 'text', 'source': 'HuMengqing.pdf'}
To verify the effectiveness of the ResNet-V2 model, comparative experiments are conducted with other classical convolutional neural network models, including EfficientNet-B0 and VGG16. The experimental results show that the ResNet26-V2 model not only outperforms the other models in terms of classifi

2. HuMengqing_chunk_126 | Rerank: 0.154476
RRF: 0.032787
{'source': 'HuMengqing.pdf', 'chunk_index': 126, 'chunk_type': 'text', 'page': 63, 'section_title': '4.4 Comparison with Other Convolutional Neural Networks'}
By comparing the results, it is found that ResNet26-V2 performs better in each metric relative to EfficientNet-B0 and VGG16. In addition, the ResNet26-V2 model designed in this study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary classification tasks, ha